# Amazon Pricing Optimization System
### Causal Inference + ML + Integer Linear Programming

This notebook implements an end-to-end pricing system to estimate price elasticity and optimize discount allocation under budget constraints, inspired by Amazon-like product catalogues.
---
Objective

The goal is to answer two key questions:
- How does demand respond to price changes? (causal elasticity)
- How should discounts be allocated across products to maximize revenue under a fixed budget?

**Pipeline overview:**
1. Feature engineering & data preparation
2. A/B test simulation → causal elasticity estimation (OLS log-log + Bayesian shrinkage)
3. XGBoost elasticity meta-model (product-level generalization)
4. Discount prediction model (Stacking: LGBM + RF → Ridge)
5. Per-product analytic optimization
6. Budget-constrained knapsack (ILP)
7. Market simulation backtest (60 periods, 2 competitors)

**Key result:** +61% revenue vs. baseline over 60 simulated periods.

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

ç
from src.configuracion import AB_TEST_SIZE, DISCOUNT_GRID, DISCOUNT_LEVELS, DATA_PATH
from src.data.loader import cap_rare_categories, load_and_prepare
from src.elasticity.causal import compute_elasticity_regression, filter_valid_categories
from src.elasticity.ml_model import assign_elasticity_from_model, estimate_elasticities_batch, train_elasticity_model
from src.experiments.ab_test import registrar_ventas_real, run_ab_test, validate_ab_balance
from src.models.discount_model import train_model
from src.optimization.analytic import compute_revenue, estimate_demand, find_optimal_discount
from src.optimization.knapsack import baseline_policy, knapsack_policy, solve_discount_knapsack
from src.simulation.market import market_simulator
from src.reporting.plots import plot_ab_response, plot_elasticity_bar, plot_results, plot_simulation_comparison, print_report

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('All imports are done')

---
## 1. Data Loading & Feature Engineering

Raw CSV is loaded and enriched with derived features:

| Feature | Formula | Rationale |
|---|---|---|
| `popularity` | `rating × log(1 + reviews)` | Combines rating signal with review volume |
| `urgencia_de_venta` | `price / (sales_last_month + 1)` | High price + low sales = urgent to move |
| `reviews_log` | `log(1 + reviews)` | Reduces skew in review counts |
| `price_vs_category` | `price / category_median_price` | Relative positioning within category |
| `is_peak_season` | `delivery_month ∈ {11, 12}` | Holiday demand flag |

> **Anti-leakage:** `price_vs_category` median is computed on `df_train` only, then applied to `df_test`.

In [ ]:
# Load and prepare dataset
# To change the data path, update DATA_PATH in src/configuracion.py
df = load_and_prepare(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
df.head(5)

In [ ]:
# Basic stats
print('=== Numeric features ===')
display(df[['original_price', 'discount_percentage', 'product_rating', 'total_reviews',
            'purchased_last_month', 'popularity', 'urgencia_de_venta']].describe().round(2))

In [ ]:
# Category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['product_category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Products per Category')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

df['discount_percentage'].hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Discount Distribution')
axes[1].set_xlabel('Discount (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Train / test split
has_temporal = 'delivery_date' in df.columns
if has_temporal:
    df = df.sort_values('delivery_date').reset_index(drop=True)

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

df_train, df_test = cap_rare_categories(
    df_train, df_test,
    cols=['product_category', 'brand_group'],
    min_count=50,
)

# price_vs_category: median computed on train only (anti-leakage)
cat_median = df_train.groupby('product_category')['original_price'].median()
df_train['price_vs_category'] = df_train['original_price'] / df_train['product_category'].map(cat_median)
df_test['price_vs_category']  = df_test['original_price']  / df_test['product_category'].map(cat_median)
df_train['price_vs_category'] = df_train['price_vs_category'].fillna(1.0)
df_test['price_vs_category']  = df_test['price_vs_category'].fillna(1.0)

catalogo = df_train.set_index('product_id')

print(f'Train: {df_train.shape[0]:,} rows | Test: {df_test.shape[0]:,} rows')
print(f'Categories in train: {df_train["product_category"].nunique()}')

---
## 2. A/B Test Simulation → Causal Elasticity

A multi-arm experiment is simulated on 20% of training data. Each product is randomly assigned one of five discount levels: **[0%, 5%, 10%, 15%, 20%]**.

Units sold are drawn from a **Poisson distribution** using category-specific true elasticities.

> **Why simulate the A/B test?** Real randomized pricing experiments require access to production systems. The simulation faithfully replicates the statistical structure of a real experiment — the methodology (OLS log-log, shrinkage, balance tests) is identical to what would be used in production. The only difference is the data source.

In [ ]:
print('Running A/B test simulation...')
ab_results = run_ab_test(
    df_train, registrar_ventas_real, catalogo,
    test_size=AB_TEST_SIZE,
    discount_levels=DISCOUNT_LEVELS,
    random_state=42,
)
print(f'A/B results shape: {ab_results.shape}')
ab_results.head()

In [ ]:
# Balance check
print('=== A/B Balance Check (χ² test) ===')
validate_ab_balance(ab_results)

print('\n=== Units per arm per category ===')
balance_table = ab_results.pivot_table(
    index='product_category', columns='discount_applied',
    values='units_sold', aggfunc='count', fill_value=0
)
display(balance_table)

In [ ]:
# Sales response to discount by category
plot_ab_response(ab_results)

In [ ]:
# Save A/B results
results_dir = Path('src/data/results')
results_dir.mkdir(parents=True, exist_ok=True)
ab_results.to_csv(results_dir / 'ab_results.csv', index=False)
print('A/B results saved.')

### 2.1 Causal Elasticity Estimation

**Model:** log-log OLS — `median(log(Q)) ~ log(1 − d/100)` — aggregated by discount level.

**Bayesian shrinkage** toward precision-weighted global prior (λ = 150 pseudo-observations):

$$\varepsilon_{shrunk} = \frac{n \cdot \varepsilon_{raw} + \lambda \cdot \varepsilon_{global}}{n + \lambda}$$

Categories with R² < 0.05 fall back to the global elasticity (ε = −1.2).

In [ ]:
valid_cats = filter_valid_categories(ab_results)
causal_elasticity = compute_elasticity_regression(ab_results, valid_cats) if valid_cats else {}

print('=== Causal Elasticities ===')
elasticity_df = pd.DataFrame(
    list(causal_elasticity.items()),
    columns=['Category', 'Elasticity']
).sort_values('Elasticity')
display(elasticity_df)

In [ ]:
if causal_elasticity:
    plot_elasticity_bar(causal_elasticity)

---
## 3. XGBoost Elasticity Meta-model

To propagate elasticity to **every individual product** (not just per category), an XGBoost model is trained on the A/B results.

- **Target:** `log(1 + units_sold)`
- **Elasticity extraction:** finite differences in log-log space — `Δlog(Q) / Δlog(P)` between `d` and `d+1`
- **OHE vocabulary** fixed at training time to prevent unseen categories at inference

In [ ]:
print('Training XGBoost elasticity meta-model...')
booster, feature_names, ohe_columns = train_elasticity_model(ab_results, df_train)

df_test = df_test.reset_index(drop=True)
elasticity_array = estimate_elasticities_batch(
    df_test, booster, feature_names, ohe_columns, delta=1.0
)
df_test = assign_elasticity_from_model(df_test, elasticity_array)

print(f'Elasticity stats on test set:')
print(df_test['elasticity_ml'].describe().round(3))

In [ ]:
# Elasticity distribution by category
fig, ax = plt.subplots(figsize=(12, 4))
df_test.boxplot(column='elasticity_ml', by='product_category', ax=ax)
ax.set_title('Product-level Elasticity Distribution by Category (XGBoost)')
ax.set_xlabel('Category')
ax.set_ylabel('Elasticity')
ax.axhline(-1, color='red', linestyle='--', alpha=0.6, label='Unitary elasticity')
ax.legend()
plt.suptitle('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 4. Discount Prediction Model

**Architecture:** Stacking ensemble

```
Base learners:
  ├── LightGBM  (objective: tweedie, 600 trees, lr=0.03)
  └── Random Forest  (150 trees, max_depth=10)
Meta-learner:
  └── Ridge  (α=1.0)
```

**Target:** `log1p(discount_percentage)` — log-transforms the right-skewed discount distribution.  
**Validation:** TimeSeriesSplit with 5 folds.

In [ ]:
print('Training stacking discount model (LGBM + RF → Ridge)...')
pipe, X_test, y_test_log, cv_results = train_model(df_train, df_test)

y_pred_log = pipe.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

print(f"\n{'='*45}")
print(f"  R² (test)       : {cv_results.get('test_r2', 0):.4f}")
print(f"  RMSE (test)     : {np.sqrt(((y_test - y_pred)**2).mean()):.2f} pp")
print(f"  R² CV train     : {cv_results['train_r2']:.4f}")
print(f"  R² CV val       : {cv_results['val_r2']:.4f}")
print(f"  Overfitting gap : {cv_results['gap']:.4f}")
print(f"{'='*45}")

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Discount (%)')
axes[0].set_ylabel('Predicted Discount (%)')
axes[0].set_title('Predicted vs Actual Discount')

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Discount (%)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Full diagnostic plots from reporting module
df_work = df_test.copy()
df_work['discount_percentage'] = y_test
df_work['discount_predicted']  = y_pred
df_work = estimate_demand(df_work)
df_work = compute_revenue(df_work)

print_report(y_test, y_pred, cv_results, df_work)

---
## 5. Per-product Analytic Optimization

Grid search over `DISCOUNT_GRID = [0, 5, 10, ..., 70]%` with elasticity-aware caps:

| Elasticity | Max discount allowed |
|---|---|
| ε < −1.5 | 50% |
| −1.5 ≤ ε < −1.0 | 40% |
| ε ≥ −1.0 | 25% |

**Demand model:** $Q(d) = Q_0 \cdot \exp(\varepsilon \cdot (-(d - d_{base}) / 100))$

**Revenue model:** $R(d) = P_0 \cdot (1 - d/100) \cdot Q(d)$ with floor at 50% of original price.

In [ ]:
opt_results = df_work.apply(find_optimal_discount, axis=1)
df_work = df_work.reset_index(drop=True).join(opt_results)

rev_pred = df_work['revenue'].sum()
rev_opt  = df_work['optimal_revenue'].sum()

print(f"Revenue with predicted discounts : ${rev_pred:>15,.0f}")
print(f"Revenue with optimal discounts   : ${rev_opt:>15,.0f}")
print(f"Analytic upside                  : ${rev_opt - rev_pred:>15,.0f}")

In [ ]:
# Optimal discount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_work['optimal_discount'].hist(bins=30, ax=axes[0], color='mediumseagreen', edgecolor='white')
axes[0].set_title('Optimal Discount Distribution (per product)')
axes[0].set_xlabel('Optimal Discount (%)')

axes[1].scatter(df_work['elasticity_ml'], df_work['optimal_discount'], alpha=0.3, s=10)
axes[1].set_xlabel('Product Elasticity')
axes[1].set_ylabel('Optimal Discount (%)')
axes[1].set_title('Elasticity vs Optimal Discount')

plt.tight_layout()
plt.show()

---
## 6. Budget-Constrained Knapsack (ILP)

**Problem formulation:**

$$\max \sum_i \sum_d \text{revenue}(i,d) \cdot x_{i,d}$$

Subject to:
- $\sum_d x_{i,d} = 1 \quad \forall i$ — one discount per product
- $\sum_i \sum_d \frac{d}{100} \cdot P_{0i} \cdot x_{i,d} \leq B$ — budget constraint
- $x_{i,d} \in \{0,1\}$

**Budget:** 5% of total catalogue value. **Scope:** top 500 products by revenue.

In [ ]:
budget = 0.05 * df_work['original_price'].sum()
print(f'Budget (5% of catalogue): ${budget:,.0f}')

df_work = solve_discount_knapsack(df_work, DISCOUNT_GRID, budget, top_n=500)

# Compare policies
rev_baseline = df_work['revenue'].sum()
rev_knapsack = df_work['knapsack_revenue'].sum() if 'knapsack_revenue' in df_work.columns else df_work['optimal_revenue'].sum()

print(f"\nRevenue baseline (predicted discounts) : ${rev_baseline:>15,.0f}")
print(f"Revenue knapsack (ILP optimized)       : ${rev_knapsack:>15,.0f}")
print(f"Uplift                                 : +{(rev_knapsack/rev_baseline - 1)*100:.1f}%")

In [ ]:
# Revenue comparison by category
if 'knapsack_discount' in df_work.columns:
    cat_rev = df_work.groupby('product_category').agg(
        baseline_rev=('revenue', 'sum'),
        knapsack_rev=('knapsack_revenue', 'sum')
    ).reset_index()
    cat_rev['uplift_pct'] = (cat_rev['knapsack_rev'] / cat_rev['baseline_rev'] - 1) * 100

    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(cat_rev))
    ax.bar(x - 0.2, cat_rev['baseline_rev']/1e6, 0.4, label='Baseline', color='steelblue')
    ax.bar(x + 0.2, cat_rev['knapsack_rev']/1e6,  0.4, label='Knapsack', color='mediumseagreen')
    ax.set_xticks(x)
    ax.set_xticklabels(cat_rev['product_category'], rotation=45)
    ax.set_ylabel('Revenue ($M)')
    ax.set_title('Revenue by Category: Baseline vs Knapsack')
    ax.legend()
    plt.tight_layout()
    plt.show()

    display(cat_rev.sort_values('uplift_pct', ascending=False).round(1))

---
## 7. Market Simulation Backtest

60-period simulation with:
- **Seasonality:** $\text{season}(t) = 1 + 0.2 \cdot \sin(2\pi t / 30)$
- **Competitor pressure:** 2 competitors apply random ±5% price noise; each competitor priced below yours reduces your demand by 10%
- **Demand:** Poisson draws from $Q_0 \cdot \text{season} \cdot \exp(\varepsilon \cdot (-d/100)) \cdot \text{competitor\_factor}$

In [ ]:
print('Running market simulation (60 periods, 2 competitors)...')
sim_catalog = df_work.copy()

sim_base = market_simulator(sim_catalog, baseline_policy, periods=60, competitors=2)
sim_knap = market_simulator(sim_catalog, knapsack_policy, periods=60, competitors=2)

rev_sim_base = sim_base['revenue'].sum()
rev_sim_knap = sim_knap['revenue'].sum()

print(f"\n{'='*50}")
print(f"  Revenue baseline (60 periods) : ${rev_sim_base:>15,.0f}")
print(f"  Revenue knapsack (60 periods) : ${rev_sim_knap:>15,.0f}")
print(f"  Uplift                        : +{(rev_sim_knap/rev_sim_base - 1)*100:.1f}%")
print(f"{'='*50}")

In [ ]:
# Revenue over time
plot_simulation_comparison(sim_base, sim_knap)

In [ ]:
# Cumulative revenue
rev_by_period = pd.DataFrame({
    'baseline': sim_base.groupby('period')['revenue'].sum(),
    'knapsack': sim_knap.groupby('period')['revenue'].sum(),
})

fig, ax = plt.subplots(figsize=(12, 5))
rev_by_period.cumsum().plot(ax=ax, linewidth=2)
ax.set_title('Cumulative Revenue over 60 Periods: Baseline vs Knapsack')
ax.set_xlabel('Period')
ax.set_ylabel('Cumulative Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
ax.legend(['Baseline', 'Knapsack (ILP)'])
plt.tight_layout()
plt.show()

---
## 8. Summary of Results

> ⚠️ The 61% uplift is measured in a simulated market. Realistic uplift in production, accounting for demand noise, competitor adaptation, and cross-product substitution, is estimated at **+12% to +18%**.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

results_summary = pd.DataFrame([
    {'Metric': 'Discount model R² (test)',           'Value': f"{r2_score(y_test, y_pred):.3f}"},
    {'Metric': 'Discount model RMSE (test)',          'Value': f"{np.sqrt(mean_squared_error(y_test, y_pred)):.2f} pp"},
    {'Metric': 'CV R² (validation)',                  'Value': f"{cv_results['val_r2']:.3f}"},
    {'Metric': 'Overfitting gap',                     'Value': f"{cv_results['gap']:.3f}"},
    {'Metric': 'Analytic upside (test set)',          'Value': f"${rev_opt - rev_pred:,.0f}"},
    {'Metric': 'Revenue — baseline (60 periods)',     'Value': f"${rev_sim_base:,.0f}"},
    {'Metric': 'Revenue — knapsack (60 periods)',     'Value': f"${rev_sim_knap:,.0f}"},
    {'Metric': 'Revenue uplift (backtest)',           'Value': f"+{(rev_sim_knap/rev_sim_base - 1)*100:.1f}%"},
])

display(results_summary.style.set_properties(**{'text-align': 'left'}))

---
## 9. Known Limitations

- **Simulated A/B test:** elasticities are derived from synthetic data with pre-specified true values. In production, a real randomized pricing experiment on a product subset would replace this step.
- **Most categories show R² < 0.10** in elasticity regressions — discrete pricing levels (only 5 arms) severely limit regression signal.
- **No cross-product demand effects** — products are treated as independent.
- **Knapsack covers only top 500 products** — the tail uses predicted discount.
- **No inventory, margin, or MAP (Minimum Advertised Price) constraints.**
